In [22]:
import os
import subprocess
import numpy as np
import pandas as pd
import seaborn as sns

In [23]:
def find_files(directory):
    for root, _, files in os.walk(directory):
        for file in files:
            yield os.path.join(root, file)


# ===== 修改为 decompress.ipynb 生成的 chunk 目录 =====
directory_to_search = '/home/user/data/partial_data_ubuntu_512KiB/'
all_files = list(find_files(directory_to_search))

print(f'chunk number: {len(all_files)}')


chunk number: 3584


In [24]:
print(all_files[0])

/home/user/data/partial_data_ubuntu_512KiB/ubuntu_lunar-20230415_layer_33_block_26.bin


# 1. LSH-based MLE

## Multiprocess to calculate

In [25]:
import os
import subprocess
import numpy as np
import pandas as pd
import seaborn as sns

import subprocess
import pandas as pd
from multiprocessing import Pool

def find_files(directory):
    for root, _, files in os.walk(directory):
        for file in files:
            yield os.path.join(root, file)


# directory_to_search = './uncompressed_data_ubuntu_4KiB/'
# all_files = list(find_files(directory_to_search))

print(f'chunk number: {len(all_files)}')


def process_file(file):
    status, output = subprocess.getstatusoutput('./finesse ' + file + ' 48 3 12')
    output = output.split('\n')
    h = output[0].split(': ')[-1]
    sf1 = output[1]
    sf2 = output[2]
    sf3 = output[3]
    return file, h, sf1, sf2, sf3

# def process_main():
all_files = list(find_files(directory_to_search))  
print(f"Total files: {len(all_files)}")
pool_size = 30  # Adjust based on your system capability

sfs = []
with Pool(pool_size) as p:
    for i, result in enumerate(p.imap(process_file, all_files), 1):
        sfs.append(result)
        if i % 10000 == 0:
            print(f"Processed {i}/{len(all_files)} files")

print('\nFinish parallel computing')
sfs_pd = pd.DataFrame(sfs, columns=['file name', 'h', 'sf1', 'sf2', 'sf3'])

# print(sfs_pd)

# sfs_pd.to_pickle('./sfs_pd_partial_data_pytorch_512KiB.pkl')

# process_main()

chunk number: 3584
Total files: 3584



Finish parallel computing


In [14]:
sfs_pd.to_pickle('./sfs_pd(partial_data_ubuntu).pkl')

In [12]:
sfs_pd.describe()

,file name,h,sf1,sf2,sf3
count,3584,3584,3584,3584,3584
unique,3584,2989,1704,1624,1368
top,/home/user/data/partial_data_ubuntu_512KiB/ubu...,812372314781133781,3568976612918044448,6849166359721366723,-6501048734054152159
freq,1,3,13,15,19


In [13]:
sfs_pd['sf3'].value_counts()

sf3
-6501048734054152159    19
5689966320291996564     18
-1142809568002161927    17
-7264128716306727247    16
6131137648532884986     15
                        ..
2766077846710300591      1
4258644391146202415      1
197047752082090869       1
-3631135908072267659     1
-7316550525581742857     1
Name: count, Length: 1368, dtype: int64

In [16]:

sf3_counts = sfs_pd['sf3'].value_counts()  

sf3_counts = sf3_counts - 1  
  
 
total = sf3_counts.sum()  
  
# Print the finesse results 
print("The total sum after subtracting 1 from each count is: ", total) 
print(total*512/1024/1024)   

The total sum after subtracting 1 from each count is:  48480
23.671875


# Use fastcdc for the finesse results

In [15]:
new_df = sfs_pd.groupby('sf3')['file name'].agg(lambda x: list(x)).reset_index()  
new_df.columns = ['sf3', 'file name'] 
new_df

,sf3,file name
0,-1013891271140676130,[/home/user/data/partial_data_ubuntu_512KiB/ub...
1,-1015925232259320187,[/home/user/data/partial_data_ubuntu_512KiB/ub...
2,-1046086924293833932,[/home/user/data/partial_data_ubuntu_512KiB/ub...
3,-1055869152482771371,[/home/user/data/partial_data_ubuntu_512KiB/ub...
4,-1070130410644024007,[/home/user/data/partial_data_ubuntu_512KiB/ub...
...,...,...
1363,92775482735558017,[/home/user/data/partial_data_ubuntu_512KiB/ub...
1364,932069116995599158,[/home/user/data/partial_data_ubuntu_512KiB/ub...
1365,966072651097794167,[/home/user/data/partial_data_ubuntu_512KiB/ub...
1366,970997905746111340,[/home/user/data/partial_data_ubuntu_512KiB/ub...


In [16]:
new_df = new_df[new_df['file name'].apply(lambda x: len(x) > 1)]  
new_df = new_df.reset_index(drop=True)  
new_df

,sf3,file name
0,-1015925232259320187,[/home/user/data/partial_data_ubuntu_512KiB/ub...
1,-1046086924293833932,[/home/user/data/partial_data_ubuntu_512KiB/ub...
2,-1114725842920054983,[/home/user/data/partial_data_ubuntu_512KiB/ub...
3,-1136049125181639519,[/home/user/data/partial_data_ubuntu_512KiB/ub...
4,-1142809568002161927,[/home/user/data/partial_data_ubuntu_512KiB/ub...
...,...,...
774,9111668364512978263,[/home/user/data/partial_data_ubuntu_512KiB/ub...
775,919841604400590499,[/home/user/data/partial_data_ubuntu_512KiB/ub...
776,92775482735558017,[/home/user/data/partial_data_ubuntu_512KiB/ub...
777,932069116995599158,[/home/user/data/partial_data_ubuntu_512KiB/ub...


In [17]:
import multiprocessing
from fastcdc import fastcdc
from hashlib import sha256

num_process=30

def process_file(file_name):
    # status, output = subprocess.getstatusoutput('fastcdc -mi 256 -s 512 -ma 1024 ' + file_name)
    # output = output.split('\n')
    results = []
    output = list(fastcdc(file_name, 256, 512, 1024, fat=True, hf=sha256))
    for i in range(len(output)):
        results.append([file_name, output[i].hash, output[i].length])
    # for line in output:
    #     temp = line.split(' ')
    #     temp_hash = temp[0].split('=')[-1]
    #     temp_offset = temp[1].split('=')[-1]
    #     temp_size = temp[2].split('=')[-1]
    #     results.append([file_name, temp_hash, temp_size])
    return results

def get_reduce_ration(fastcdc_df):

    fastcdc_df['size'] = fastcdc_df['size'].astype(int)

    # Calculate the total size of each hash value
    hash_total_size = fastcdc_df.groupby('hash')['size'].sum()

    # Calculate the number of occurrences of each hash
    hash_counts = fastcdc_df['hash'].value_counts()
    hashes_more_than_once = hash_counts[hash_counts > 1].index
    duplicated_first_instances = fastcdc_df[fastcdc_df['hash'].isin(hashes_more_than_once)].drop_duplicates('hash')
    size_sum_duplicated_first = duplicated_first_instances['size'].sum()
    
    hashes_once_only = hash_counts[hash_counts == 1].index
    instances_once_only = fastcdc_df[fastcdc_df['hash'].isin(hashes_once_only)]
    size_sum_once_only = instances_once_only['size'].sum()
    
    # Calculate the redundant size: for each hash, its total size minus the size of the first instance
    reduced_size = (hash_total_size - fastcdc_df.drop_duplicates('hash').set_index('hash')['size']).sum()
    
    
    # Calculate the total size
    total_size = hash_total_size.sum()

    # print(f"reduce size is {reduced_size}")
    # print(f"all size is {total_size}")
    return reduced_size, total_size, size_sum_duplicated_first, size_sum_once_only

def get_cluster(cluster_id):
    fastcdc_list = []
    cluster_files = new_df.iloc[cluster_id].at['file name']

    with multiprocessing.Pool(processes=num_process) as pool:
        results = pool.map(process_file, cluster_files)
        for result in results:
            fastcdc_list.extend(result)

    fastcdc_df = pd.DataFrame(fastcdc_list, columns=['file name', 'hash', 'size'])
    return fastcdc_df


all_cluster_size = []
all_cluster_reducesize = []
all_cluster_sum_duplicated_first = []
all_cluster_sum_once_only = []

for cluster_id in range(new_df.shape[0]):
    # print(f"cluster:{cluster_id}")
    temp_df = get_cluster(cluster_id)
    temp_reduce, temp_all, temp_size_sum_duplicated_first, temp_size_sum_once_only = get_reduce_ration(temp_df)
    print(f"cluster:{cluster_id}, reduced size:{temp_reduce}, cluster total size: {temp_all}, duplicated first: {temp_size_sum_duplicated_first}, once_only: {temp_size_sum_once_only}")
    all_cluster_size.append(temp_all)
    all_cluster_reducesize.append(temp_reduce)
    all_cluster_sum_duplicated_first.append(temp_size_sum_duplicated_first)
    all_cluster_sum_once_only.append(temp_size_sum_once_only)

total = sum(all_cluster_size)
total_reduce = sum(all_cluster_reducesize)
total_duplicated_first = sum(all_cluster_sum_duplicated_first)
total_once_only = sum(all_cluster_sum_once_only)

print("{} {} {}".format(total_reduce, total, total_reduce * 1.0 / total))
print(f'total reduced: {total_reduce/1024/1024/1024}GB')
print(f'total: {total/1024/1024/1024}GB')
print(f'total duplicated first: {total_duplicated_first/1024/1024/1024}GB')
print(f'total once only: {total_once_only/1024/1024/1024}GB')

cluster:0, reduced size:1114679, cluster total size: 1572864, duplicated first: 363594, once_only: 94591
cluster:1, reduced size:1523893, cluster total size: 2097152, duplicated first: 513961, once_only: 59298
cluster:2, reduced size:530666, cluster total size: 1048576, duplicated first: 517910, once_only: 0
cluster:3, reduced size:1572417, cluster total size: 2097152, duplicated first: 524288, once_only: 447
cluster:4, reduced size:8134764, cluster total size: 8912896, duplicated first: 598133, once_only: 179999
cluster:5, reduced size:485079, cluster total size: 1048576, duplicated first: 485079, once_only: 78418
cluster:6, reduced size:2463393, cluster total size: 3145728, duplicated first: 626318, once_only: 56017
cluster:7, reduced size:2682540, cluster total size: 3145728, duplicated first: 402799, once_only: 60389
cluster:8, reduced size:1017038, cluster total size: 1572864, duplicated first: 513654, once_only: 42172
cluster:9, reduced size:1557808, cluster total size: 2097152, 

# 2. Message-locked encryption (MLE)

In [18]:
import hashlib
def hash_file(filename):
    sha256_hash = hashlib.sha256()
    with open(filename, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

sha256_list = [[], []]
cnt = 0
for file in all_files:
    # status, output = subprocess.getstatusoutput('sha256sum ' + file)
    output = hash_file(file)
    cnt += 1
    print(cnt, end='\r')
    # output = output.split('  ')
    sha256_list[0].append(file)
    sha256_list[1].append(output)

ssha256_pd=pd.DataFrame(sha256_list)
ssha256_pd = ssha256_pd.T
ssha256_pd.rename(columns={0:'file name', 1:'sha256'},inplace=True)
ssha256_pd

,file name,sha256
0,/home/user/data/partial_data_ubuntu_512KiB/ubu...,34c9972bd555d2cac146d3de4deee757578efe93023843...
1,/home/user/data/partial_data_ubuntu_512KiB/ubu...,7be8f579123b64f988dbd4891cf4838bfe5dae93f877bf...
2,/home/user/data/partial_data_ubuntu_512KiB/ubu...,c547b655218622af4d876708dcaed3e30a39533eaf561a...
3,/home/user/data/partial_data_ubuntu_512KiB/ubu...,964bd39d1decab6749a00cca9866633f9f9050f91aeecb...
4,/home/user/data/partial_data_ubuntu_512KiB/ubu...,4c7e0d70dd23ddb8a9d34edde65e68475155a68e99ae95...
...,...,...
3579,/home/user/data/partial_data_ubuntu_512KiB/ubu...,d27dfa8e1a6d5535069d686d08a0954f692ad7d78c73eb...
3580,/home/user/data/partial_data_ubuntu_512KiB/ubu...,3c847cbfb206502eef05ae3dfdf73a2e1193102727d27f...
3581,/home/user/data/partial_data_ubuntu_512KiB/ubu...,909a9f9b6eb3f681c3593128677a1f3a70468d1fdf9e07...
3582,/home/user/data/partial_data_ubuntu_512KiB/ubu...,42936b926d8d92a604c254c1a64676339456b0138f33b6...


In [19]:
ssha256_pd.describe()

,file name,sha256
count,3584,3584
unique,3584,2989
top,/home/user/data/partial_data_ubuntu_512KiB/ubu...,92344ffeb19494eed14b741489532bd14ab5eae6d87d15...
freq,1,3


In [21]:
ssha256_pd['sha256'].value_counts()

sha256
92344ffeb19494eed14b741489532bd14ab5eae6d87d1583fae2ec4e2f7daf67    3
179de26b1794d57208fa7d733cde7a6898e570fedce800c5d9d4fcf75aea934e    3
e19f66d8d6dd37759f4d867fb5375a13b71647c0c8a8ba8041a563b7ee5457b3    3
5edda385061b63907ae0d3d06fa3ab490286944e393fefa6bb0cc74e418023f2    3
22cb9c343527c70598e42d950b7fe6f06933a5750bc60320b2aedceb778ed81b    3
                                                                   ..
8821bcaf25c252ee6804f3273a37b6d28704d26779071e5cb70c218c4d9302ea    1
988358ce3a3a19153dfbe3f5219575a229165d718f06e2710b9ff6c477320330    1
81537e840dcbf47e83f175111091dac521f7452eb56b5512def7ac17bf17c54e    1
d4de83e7b016ba4c9821068b35f6fc4f8bbe3851cd29abbfcd1dac7a0b650de0    1
aa84b3b6ed69301bf057dc2fa71223d3c15bb5222b85059fd5aa3514e4740727    1
Name: count, Length: 2989, dtype: int64

In [8]:
# Group by 'sha256' and calculate the total size of the duplicated files  
ssha256_pd['file size'] = ssha256_pd['file name'].apply(os.path.getsize)  
grouped = ssha256_pd.groupby('sha256').apply(lambda x: (len(x) - 1) * x['file size'].iloc[0] if len(x) > 1 else 0)  
  
# Calculate the total size of all the duplicated files  
total_deduplicated_size = grouped.sum()  
print(f'{total_deduplicated_size/1024/1024/1024} GiB,  {total_deduplicated_size}')
    

0.29052734375 GiB,  311951360


/tmp/ipykernel_1947794/2430996357.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = ssha256_pd.groupby('sha256').apply(lambda x: (len(x) - 1) * x['file size'].iloc[0] if len(x) > 1 else 0)


In [9]:
import os
import pandas as pd

ssha256_pd['file size'] = ssha256_pd['file name'].apply(os.path.getsize)

first_file_sizes = ssha256_pd.groupby('sha256').apply(lambda x: x['file size'].iloc[0] if len(x) > 1 else 0)
total_first_file_sizes = first_file_sizes.sum()

total_file_size = ssha256_pd['file size'].sum()
unique_files = ssha256_pd.groupby('sha256').filter(lambda x: len(x) == 1)
total_unique_file_size = unique_files['file size'].sum()


print(f'Total size of files with a unique sha256 hash: {total_unique_file_size/1024/1024/1024} GiB, {total_unique_file_size} bytes')

print(f'Total size of the first files in each duplicate group: {total_first_file_sizes/1024/1024/1024} GiB, {total_first_file_sizes} bytes')
print(f'Total size of all files: {total_file_size/1024/1024/1024} GiB, {total_file_size} bytes')


Total size of files with a unique sha256 hash: 1.23974609375 GiB, 1331167232 bytes
Total size of the first files in each duplicate group: 0.2197265625 GiB, 235929600 bytes
Total size of all files: 1.75 GiB, 1879048192 bytes


/tmp/ipykernel_1947794/1618021712.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_file_sizes = ssha256_pd.groupby('sha256').apply(lambda x: x['file size'].iloc[0] if len(x) > 1 else 0)
